# Network-only development checkpoint

**Recorded outcome:** the repeater-template branch passes its leakage-safe injection gate; the generic four-station trigger fails the predeclared SNR 1.0 gate. This is a development STOP checkpoint, not held-out performance and not a DAS result.

The notebook opens only compact tracked CSV/JSON products. It does not open raw network or DAS waveforms, refresh catalogs, modify thresholds, or access the 12 sealed held-out hours.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

def find_project():
    candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for candidate in candidates:
        if (candidate / "config" / "development_detection.json").exists():
            return candidate.resolve()
    raise FileNotFoundError("Run from repeaters_v2 or its notebooks directory")

PROJECT = find_project()
DEVELOPMENT = PROJECT / "outputs" / "development_network"
print("Project:", PROJECT)
print("Raw waveform access: disabled in this notebook")

## 1. Gate status and provenance

The detector thresholds were fixed before the injections. The exact injected historical event was removed from the template bank for its trial.

In [ ]:
with (DEVELOPMENT / "status.json").open(encoding="utf-8") as handle:
    status = json.load(handle)
with (PROJECT / "config" / "development_detection.json").open(encoding="utf-8") as handle:
    config = json.load(handle)

keys = [
    "stage",
    "overall_network_detector_status",
    "development_config_sha256",
    "template_detection_threshold",
    "generic_detection_threshold",
    "template_candidate_count",
    "generic_candidate_count",
    "generic_candidate_background_catalog_unassociated_count",
    "injection_trial_count",
    "injection_acceptance_status",
    "network_only_stage_das_waveforms_opened",
    "heldout_intervals_opened",
]
display(pd.DataFrame({"value": [status[key] for key in keys]}, index=keys))
assert status["network_only_stage_das_waveforms_opened"] == 0
assert status["heldout_intervals_opened"] == 0

## 2. What the network-only candidates were

The template bank found both Parkfield catalog events. The generic trigger found those two arrivals plus one arrival from a known M2.3 Carpinteria event 197 km away. Thus the apparent third trigger is not an uncataloged local earthquake.

In [ ]:
template_candidates = pd.read_csv(DEVELOPMENT / "candidate_detections.csv", dtype={"nearest_catalog_event_id": str})
generic_candidates = pd.read_csv(DEVELOPMENT / "generic_candidate_detections.csv", dtype={"nearest_catalog_event_id": str, "background_catalog_event_id": str})

display(template_candidates[[
    "candidate_id", "origin_time", "bank_score", "threshold",
    "best_template_sequence_id", "nearest_catalog_event_id",
]])
display(generic_candidates[[
    "candidate_id", "trigger_time", "coincidence_score", "threshold",
    "background_catalog_event_id", "background_catalog_location_name",
    "background_catalog_horizontal_distance_km",
    "background_catalog_observed_delay_s",
]])

## 3. Fixed-threshold injection recovery

Amplitude SNR is the injected template RMS divided by the preceding real-noise RMS, independently for each component. SNR 0 is a noise-only control. These curves test algorithmic recovery of known waveform shapes; they do **not** establish a magnitude of completeness.

In [ ]:
summary = pd.read_csv(DEVELOPMENT / "injection_recovery_summary.csv")
target = summary.loc[summary["population"] == "target_positive"].copy()

fig, ax = plt.subplots(figsize=(8, 4.8))
for detector, group in target.groupby("detector"):
    group = group.sort_values("amplitude_snr")
    y = group["recovery_fraction"].to_numpy()
    low = group["wilson_95_low"].to_numpy()
    high = group["wilson_95_high"].to_numpy()
    ax.errorbar(
        group["amplitude_snr"], y,
        yerr=np.vstack([y - low, high - y]),
        marker="o", capsize=3, label=detector,
    )
ax.axhline(0.9, color="0.4", linestyle="--", label="predeclared 0.90 gate")
ax.axvline(1.0, color="0.7", linestyle=":", label="reference SNR 1.0")
ax.set(ylim=(-0.03, 1.05), xlabel="Injected component amplitude SNR", ylabel="Target-family recovery fraction")
ax.set_xscale("symlog", linthresh=0.25)
ax.grid(alpha=0.25)
ax.legend(loc="lower right")
plt.show()
display(target)

## 4. Advisor sandbox (does not alter the registered result)

Change the three variables below to inspect a different population, SNR, or hypothetical recovery requirement. The registered v4 gate remains target-positive, SNR 1.0, minimum 0.90, and its STOP is preserved in `status.json`.

In [ ]:
SANDBOX_POPULATION = "target_positive"
SANDBOX_REFERENCE_SNR = 1.0
SANDBOX_MINIMUM_RECOVERY = 0.90

sandbox = summary.loc[
    (summary["population"] == SANDBOX_POPULATION)
    & np.isclose(summary["amplitude_snr"], SANDBOX_REFERENCE_SNR)
].copy()
sandbox["sandbox_status"] = np.where(
    sandbox["recovery_fraction"] >= SANDBOX_MINIMUM_RECOVERY,
    "PASS", "STOP",
)
display(sandbox[["detector", "trial_count", "recovered_count", "recovery_fraction", "wilson_95_low", "wilson_95_high", "sandbox_status"]])
print("Exploratory only; no config or output was written.")

## 5. Failure anatomy at the registered SNR

Failures cluster by injected waveform and noise position. This table is the evidence needed to decide whether the generic trigger should be improved, replaced by a stronger conventional picker, or retained only as an auxiliary safety net.

In [ ]:
trials = pd.read_csv(DEVELOPMENT / "injection_recovery_trials.csv", dtype={"injected_event_id": str})
registered = trials.loc[
    (trials["injected_validation_role"] == "target_positive")
    & np.isclose(trials["amplitude_snr"], 1.0)
].copy()
failures = registered.loc[~registered["generic_recovered"]]
failure_counts = (
    failures.groupby(["injected_event_id", "injected_sequence_id"], as_index=False)
    .agg(failures=("trial_id", "size"), median_peak_score=("generic_peak_score", "median"))
    .sort_values(["failures", "injected_event_id"], ascending=[False, True])
)
display(failure_counts)
display(failures[[
    "trial_id", "injected_event_id", "position_index",
    "generic_peak_score", "generic_threshold",
    "generic_peak_trigger_residual_s",
]])

## Checkpoint decision

- Keep the repeater-template branch: it recovered 30/30 target-family injections at SNR 1.0 with self-template exclusion, 30/30 at SNR 0.25, and 0/30 zero-amplitude controls.
- Do not erase the generic STOP: it recovered 22/30 at SNR 1.0, although it reached 30/30 at SNR 2.0.
- Do not open DAS or held-out hours yet. First freeze the role of the generic branch and a time-only blind union/deduplication rule.
- A later DAS extension claim must beat the union of the template branch, the generic branch, and broader-catalog vetoes—not merely the local Parkfield catalog.